# ML-09 — Validation and Research Claim Audit

**Lane 2 — Refresh / Content Opportunity Scoring · Build+ phase.** The Week-5 model trained a learned
decline-risk queue on the March-2026 page-level slice and reported it under a **client-grouped holdout** — already
the honest split on the *grouping* axis. This notebook does the next level of rigor on that same work:

1. **Two findings from the FlyRank research paper** (`docs/flyrank-seo-research-march-2026.pdf`) and the
   methodology question I would ask about each — where does the label come from, does the validation design
   carry the claim — framed as *how to make it stronger*, not as a scalp.
2. **My model under an honest split (before/after).** Re-run the Week-5 model under a **time-aware** design
   (features strictly before the label window) plus a genuine **time split** (train on March, test on April) —
   the split that mimics deployment — and show the before/after numbers, including the row-random-split numbers
   so the memorization gap is visible. Then look at **real failure examples** on the honest split: the false
   positives and missed declines of the surviving model, with concrete rows.
3. **Leakage audit** of the final feature set — the same hunt from Week 3, on the final features: timeline
   (including a forward-dated `days_since_last_update` fix), deliberate leak injection (the "watch the score jump
   toward 1.0" test), label-sibling columns, product flags, base rate next to every metric.
4. **Claim rewrite** — my own boldest Week-5 sentences rewritten in public-safe language.

Skills loaded for this task: `hunting-leakage-and-validating` (leakage taxonomy, honest splits, base rates,
attack checklist) and `flyrank/flyrank-data` (label trap: `trend_direction`/`trend_pct` are the label, never
features; IDs for grouping only; the query table's window overlap warning). Claim language follows
`writing-honest-claims`: observed / measured / directional / decision-support.

## Setup

Same access pattern as w05: DuckDB secret with the repo `.env` HF READ token (never hardcoded), warehouse
release `v20260703`, one row per page per decision moment.

In [1]:
%pip install -q duckdb huggingface_hub python-dotenv scikit-learn pandas matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: C:\Program Files\Python313\python.exe -m pip install --upgrade pip


In [2]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd

from dotenv import load_dotenv

load_dotenv("../../.env")
HF_TOKEN = os.environ.get("HF_token") or os.environ.get("HF_TOKEN")
assert HF_TOKEN, "No HF token found in .env"

import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT_MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
FACT_APR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"
DIM = f"read_parquet('{REL}/dim_content.parquet')"

OUT_DIR = Path("../../").resolve() / "work" / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Two paper findings + my methodology questions

Read: `docs/flyrank-seo-research-march-2026.pdf` (36 pp, 341,701 content pieces / 57 brands). I applied the
same three questions to the paper that this notebook applies to my own Week-5 model: **where does the label
come from, what does the validation design actually test, would the number survive a grouped or time split?**
Both are framed constructively — the paper already holds itself to disclosed standards (it demotes weak cuts,
flags survivor bias, and calls its ML appendix "descriptive"), so the questions are about reaching the next
level of rigor, not grading it.

### Finding A — "The Freshness Multiplier" (paper Finding #4, p.9)

**Paper claim:** *"365+ day content refreshed within 30 days shows 3.2x health boost (10.7 to 34.5) and 57x
more impressions (71 to 4,039); the 31-90 day freshness band is the strongest measured window at 7.88:1
growth-to-decline."*

**The methodology questions I would ask:**

1. **Where does the label come from?** "Growing/declining" is the impressions trend (30d vs previous 30d), and
   the "health" being boosted is FlyRank's composite — impressions (30) + position (30) + CTR (20) + scroll (20).
   Part of the "3.2x health boost" is therefore re-measuring the impressions and position that *define* the
   score, and the 57x impression comparison is between two different page populations (refreshed vs untouched),
   not a before/after of the same pages.
2. **Does the validation design carry the claim?** The refresh assignment was **chosen**, not random — someone
   picked which mature pages to update, and the picking is correlated with existing demand and visibility. A
   cross-sectional gap between *chosen* and *unchosen* pages therefore does not measure the *effect of
   refreshing*; part of the gap is the choosing. The paper models this well where it discloses the tiny 361+ cell
   (1 declining page, ratio 283:1) and refuses it as a headline — that is the exact disclosure habit I want in my
   own work.
3. **How to make it stronger:** within-page before/after over the same window, or a matched control of
   similar-age, similar-demand untouched pages, and say the selection rule in the same sentence. In safe
   language the finding becomes: *refreshed mature pages in this portfolio showed higher measured impressions
   than similar-aged untouched pages — associated with the refresh, not evidence the refresh caused the gain.*

### Finding B — ML appendix "What Predicts Growth" (p.29)

**Paper claim:** *"Logistic regression (71% holdout accuracy) describing which sampled features separate
growing from declining pages; content age is the strongest negative signal."* (Sklearn appendix: 80/20 splits,
no p-values or intervals, active-content subset with sessions > 0 and impressions > 0.)

**The methodology questions I would ask:**

1. **Where does the label come from?** Growing/declining is derived from the same impressions-based trend as
   several inputs the model reads (impressions, days visible, position) — the label and the features share a
   family, so feature importance is partly **label-derived**. The paper handles this correctly by labeling the
   appendix "descriptive, not causal"; that is the disclosure I copied into my own importance plots in Week 5.
2. **Does the validation carry the claim?** 71% accuracy is reported with **no base rate** — and the paper's own
   "evidence standard" spirit asks for one. For context the broader portfolio is ~62% growing pages (Finding #1:
   74.8K up vs 45.6K down), so "always guess growing" scores ~62% there; the ML appendix's own base rate is not
   disclosed, so the 71% should be read in that portfolio context, not at face value. And the paper never says
   whether the 80/20 split groups by brand: if pages of the same brand sit in both folds, the model can memorize
   brand patterns instead of learning a transferable rule — the question I would ask, and the exact reason my
   Week-5 model switched to a client-grouped holdout.
3. **How to make it stronger:** print the base rate next to every accuracy (which the paper's own "evidence
   standard" spirit asks for), group the split by brand, and use a time split if the outcome is trend-like.

Both findings map directly onto the three checks I run on my own model below: label-sibling features (Finding B),
selection in the split (Finding A), and base rates next to every score.

## 2. My model under an honest split (before/after)

**What the Week-5 model already did honestly:** a **client-grouped holdout** (20% of clients held out, seed 42)
so the queue is scored on clients the model never trained on — the honest *grouping* axis.

**The honest gap Week-5 disclosed:** its March features aggregate over the **whole month**, which overlaps the
second half (Mar 16-31) that *defines the label*. That overlap is window leakage — the paper's own appendix and
the warehouse query-table warning both flag it. This notebook closes that gap in two ways:

- **Time-aware features (the "after"):** features are computed over **Mar 1-15 only** — knowable at the decision
  moment and strictly before the label window — keeping the same pages, same label, same client-grouped split.
- **Genuine time split (deployment simulation):** train on the **March** decision moment, test on the **April**
  decision moment — the only split that mimics deployment for anything trend-like.

I also re-run the **row-random split** (the reference pipeline's and the paper's choice) on both feature sets so
the memorization gap (random -> grouped) and the window-overlap gap (full-month -> first-half) are both visible.
All numbers sit next to their **base rate**, always.

In [3]:
def build_month(month, fact, decision_day=15):
    """One row per page at a month's decision moment (w03/w05 contract population).

    Population rule (identical to w05): page-level rows where the full month has GSC+GA4
    data available, full-month impressions >= 100, and a non-missing full-month average
    position (the w05 dropna rule).

    Returns BOTH feature variants so the before/after comparison uses the same pages:
    - *_month       -> full-month aggregates (the w05 'as-shipped' features, incl. the overlap)
    - *_first_half  -> aggregates over day 1..15 only (time-aware: knowable at the decision
      moment, strictly before the second-half label window)
    """
    snap = f"DATE '{month}-{decision_day:02d}'"
    cut = f"DATE '{month}-{decision_day:02d}'"
    df = con.sql(
        f"""
        WITH daily AS (
            SELECT *
            FROM {fact}
            WHERE ga4_data_available IS TRUE
              AND gsc_data_available IS TRUE
        ),
        page AS (
            SELECT
                d.client_hash_id,
                d.content_hash_id,
                SUM(d.gsc_impressions) AS gsc_impressions_month,
                AVG(CASE WHEN d.gsc_avg_position > 0 THEN d.gsc_avg_position END) AS avg_position_month,
                SUM(d.ga4_sessions) AS sessions_month,
                SUM(d.ga4_engaged_sessions) AS engaged_sessions_month,
                SUM(CASE WHEN d.report_date <= {cut} THEN d.gsc_impressions ELSE 0 END) AS gsc_impressions_first_half,
                AVG(CASE WHEN d.gsc_avg_position > 0 AND d.report_date <= {cut} THEN d.gsc_avg_position END) AS avg_position_first_half,
                SUM(CASE WHEN d.report_date <= {cut} THEN d.ga4_sessions ELSE 0 END) AS sessions_first_half,
                SUM(CASE WHEN d.report_date <= {cut} THEN d.ga4_engaged_sessions ELSE 0 END) AS engaged_sessions_first_half,
                SUM(CASE WHEN d.report_date <= {cut} THEN d.gsc_impressions ELSE 0 END) AS imp_first_half,
                SUM(CASE WHEN d.report_date >  {cut} THEN d.gsc_impressions ELSE 0 END) AS imp_second_half
            FROM daily d
            GROUP BY 1, 2
            HAVING SUM(d.gsc_impressions) >= 100
        )
        SELECT
            p.client_hash_id,
            p.content_hash_id,
            DATE_DIFF('day', dc.content_created_date, {snap}) AS content_age_days,
            CASE WHEN dc.content_updated_date <= {snap}
                 THEN DATE_DIFF('day', dc.content_updated_date, {snap}) END AS days_since_last_update,
            p.gsc_impressions_month,
            p.avg_position_month,
            p.sessions_month,
            p.engaged_sessions_month,
            p.gsc_impressions_first_half,
            p.avg_position_first_half,
            p.sessions_first_half,
            p.engaged_sessions_first_half,
            p.imp_first_half,
            p.imp_second_half,
            CASE WHEN p.imp_second_half < p.imp_first_half THEN 1 ELSE 0 END AS is_declining_label
        FROM page p
        JOIN {DIM} dc USING (content_hash_id)
        """
    ).df()
    # Normalize numeric dtypes: SQL NULLs arrive as pandas nullable (pd.NA), which breaks .round()
    # downstream; coerce everything numeric to plain float64 with NaN.
    for c in ["content_age_days", "days_since_last_update", "gsc_impressions_month",
              "avg_position_month", "sessions_month", "engaged_sessions_month",
              "gsc_impressions_first_half", "avg_position_first_half",
              "sessions_first_half", "engaged_sessions_first_half",
              "imp_first_half", "imp_second_half"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype(float)
    # Identical population rule to w05: pages with a non-missing full-month average position.
    df = (df.dropna(subset=["avg_position_month"])
             .sort_values(["client_hash_id", "content_hash_id"])
             .reset_index(drop=True))
    return df


march = build_month("2026-03", FACT_MAR)
april = build_month("2026-04", FACT_APR)

print(f"March slice: {len(march):,} rows / {march['client_hash_id'].nunique()} clients | base rate {march['is_declining_label'].mean():.3f}")
print(f"April slice: {len(april):,} rows / {april['client_hash_id'].nunique()} clients | base rate {april['is_declining_label'].mean():.3f}")
print("Pages present in both months:", int(len(set(march['content_hash_id']) & set(april['content_hash_id']))))

March slice: 32,596 rows / 30 clients | base rate 0.268
April slice: 36,262 rows / 34 clients | base rate 0.479
Pages present in both months: 26036


**Feature builders and splits.** `build_ashipped` reproduces the Week-5 feature matrix verbatim (6 features, incl.
the full-month overlap and the w05 forward-date clamp on `days_since_last_update`). `build_honest` is the time-aware
version: only first-half aggregates, with explicit `has_*` flags for missing first-half position/engagement and for
an update date not yet knowable at the decision moment (a page updated after the decision date, or never updated,
is `has_days_since_update=0`, not 'freshly updated'). Splits: a `client_grouped_split` (the honest grouping axis) and
a plain `random_split` (the row-random axis used by the reference pipeline and the paper).

In [4]:
ASHIPPED = ["content_age_days", "days_since_last_update", "avg_position_month",
            "log_gsc_impressions_month", "engagement_rate_month", "has_engagement_month"]


def build_ashipped(df):
    """Week-5 'as-shipped' feature matrix, verbatim (fillna(0) reproduces the w05
    GREATEST(..., 0) clamp for post-decision update dates)."""
    X = pd.DataFrame(index=df.index)
    X["content_age_days"] = df["content_age_days"].astype(float)
    X["days_since_last_update"] = df["days_since_last_update"].fillna(0.0).astype(float)
    X["avg_position_month"] = df["avg_position_month"].astype(float)
    X["log_gsc_impressions_month"] = np.log1p(df["gsc_impressions_month"].astype(float))
    er = pd.to_numeric(df["engaged_sessions_month"] / df["sessions_month"].where(df["sessions_month"] > 0),
                       errors="coerce") * 100.0
    X["has_engagement_month"] = er.notna().astype(int)
    X["engagement_rate_month"] = er.fillna(0.0)
    return X[ASHIPPED]


HONEST = ["content_age_days", "days_since_last_update", "has_days_since_update",
          "avg_position_first_half", "has_position_first_half",
          "log_gsc_impressions_first_half", "engagement_rate_first_half",
          "has_engagement_first_half"]


def build_honest(df):
    """Time-aware feature matrix: first-half aggregates only, with an update date knowable
    at the decision moment (a post-decision or missing update date is unknown, flagged by
    has_days_since_update)."""
    X = pd.DataFrame(index=df.index)
    X["content_age_days"] = df["content_age_days"].astype(float)
    X["days_since_last_update"] = df["days_since_last_update"].astype(float).fillna(0.0)
    X["has_days_since_update"] = df["days_since_last_update"].notna().astype(int)
    X["avg_position_first_half"] = df["avg_position_first_half"].astype(float).fillna(0.0)
    X["has_position_first_half"] = df["avg_position_first_half"].notna().astype(int)
    X["log_gsc_impressions_first_half"] = np.log1p(df["gsc_impressions_first_half"].astype(float))
    er = pd.to_numeric(df["engaged_sessions_first_half"] / df["sessions_first_half"].where(df["sessions_first_half"] > 0),
                       errors="coerce") * 100.0
    X["has_engagement_first_half"] = er.notna().astype(int)
    X["engagement_rate_first_half"] = er.fillna(0.0)
    return X[HONEST]


def client_grouped_split(df, frac=0.2, seed=RANDOM_STATE):
    """Hold out `frac` of clients; rows of one client never straddle the split."""
    rng = np.random.default_rng(seed)
    clients = np.sort(df["client_hash_id"].drop_duplicates().to_numpy())
    n_test = max(1, int(round(len(clients) * frac)))
    test_clients = set(clients[rng.permutation(len(clients))[:n_test]])
    is_test = df["client_hash_id"].isin(test_clients).to_numpy()
    return ~is_test, is_test, test_clients


def random_split(df, frac=0.2, seed=RANDOM_STATE):
    """Row-random split (stratified), the reference-pipeline/paper choice."""
    from sklearn.model_selection import train_test_split
    idx = np.arange(len(df))
    tr_idx, te_idx = train_test_split(
        idx, test_size=frac, random_state=seed,
        stratify=df["is_declining_label"].astype(int).to_numpy(),
    )
    mask_train = np.zeros(len(df), dtype=bool)
    mask_train[tr_idx] = True
    mask_test = np.zeros(len(df), dtype=bool)
    mask_test[te_idx] = True
    return mask_train, mask_test, None


def split_frame(df, mask_train, mask_test):
    return df.loc[mask_train].reset_index(drop=True), df.loc[mask_test].reset_index(drop=True)

**Models and metrics** — identical hyper-parameters to the repo reference pipeline and the Week-5 notebook
(LR + depth-3 tree + RF), precision@K as the queue metric, ROC-AUC + average precision as ranking checks. The
w04 rule is re-scored on the same test rows (it has no fitted parameters). Base rate = what random ranking gives.

One disclosure: the rule has no fitted parameters, so `encode_rule` always reads its own **full-month** columns no
matter which feature variant the cell uses — its row is therefore constant across the before/after cells and, like
the Week-5 model it encodes, still carries the full-month window overlap. That makes the model-vs-rule gap in the
"after" cells conservative, not optimistic.

In [5]:
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

KS = [10, 25, 50, 100]


def precision_at_k(y, score, k):
    order = np.argsort(-np.asarray(score), kind="stable")
    return float(y[order][:k].mean())


def encode_rule(df):  # the w04 rule, verbatim
    df = df.copy()
    vis = (df["gsc_impressions_month"] >= 500).astype(int)
    depth = df["avg_position_month"].clip(upper=50) / 50.0
    slip = (df["avg_position_month"] >= 10).astype(int)
    df["score"] = vis * slip * depth
    df["queue_score"] = df["score"] + (df["gsc_impressions_month"] / df["gsc_impressions_month"].quantile(0.995)) * 1e-6
    return df


def make_models():
    return {
        "logistic_regression": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
        ]),
        "decision_tree_d3": DecisionTreeClassifier(
            class_weight="balanced", max_depth=3, min_samples_leaf=50, random_state=RANDOM_STATE
        ),
        "random_forest": RandomForestClassifier(
            class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
            n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE,
        ),
    }


def evaluate(y, score):
    has2 = len(set(np.asarray(y).tolist())) == 2
    return {f"prec@{k}": precision_at_k(y, score, k) for k in KS} | {
        "roc_auc": roc_auc_score(y, score) if has2 else float("nan"),
        "avg_precision": average_precision_score(y, score) if has2 else float("nan"),
    }


def fit_and_evaluate(df, X_builder, mask_tr, mask_te, with_rule=True):
    tr, te = split_frame(df, mask_tr, mask_te)
    X_tr, X_te = X_builder(tr), X_builder(te)
    y_tr = tr["is_declining_label"].astype(int).to_numpy()
    y_te = te["is_declining_label"].astype(int).to_numpy()
    out = {"n_test": len(te), "base_rate": float(y_te.mean())}
    if with_rule:
        rule = encode_rule(te.copy())["queue_score"].to_numpy()
        out["rule"] = evaluate(y_te, rule)
    for name, model in make_models().items():
        model.fit(X_tr, y_tr)
        out[name] = evaluate(y_te, model.predict_proba(X_te)[:, 1])
    return out

print(f"scikit-learn {sklearn.__version__} | seed {RANDOM_STATE} everywhere")

scikit-learn 1.9.0 | seed 42 everywhere


**Before/after table.** Four cells, same March pages (32,596), same label, same seed:

| Cell | Features | Split | What it shows |
|---|---|---|---|
| before-1 | full-month (w05 as shipped) | row-random | the paper's / reference pipeline's design |
| before-2 | full-month (w05 as shipped) | client-grouped | the Week-5 shipped numbers (reproduced) |
| after-1 | time-aware (first half) | row-random | window-overlap removed, still row-random |
| after-2 | time-aware (first half) | client-grouped | the honest number on both axes |

The gap from before-1 to after-2 is the improvement; the sub-gaps decompose how much of the original score was
memorization (random -> grouped) vs window overlap (full-month -> first-half).

In [6]:
configs = [
    ("before-1 full-month / random",  march, build_ashipped, random_split),
    ("before-2 full-month / grouped", march, build_ashipped, client_grouped_split),
    ("after-1  time-aware / random",  march, build_honest,   random_split),
    ("after-2  time-aware / grouped", march, build_honest,   client_grouped_split),
]

rows = []
for label, df, builder, split_fn in configs:
    m_tr, m_te, _ = split_fn(df)
    r = fit_and_evaluate(df, builder, m_tr, m_te)
    rows.append({
        "config": label,
        "n_test": r["n_test"],
        "base_rate": round(r["base_rate"], 3),
        "rule_p50": round(r["rule"]["prec@50"], 3),
        "rule_auc": round(r["rule"]["roc_auc"], 3),
        "tree_p50": round(r["decision_tree_d3"]["prec@50"], 3),
        "tree_auc": round(r["decision_tree_d3"]["roc_auc"], 3),
        "lr_p50": round(r["logistic_regression"]["prec@50"], 3),
        "rf_p50": round(r["random_forest"]["prec@50"], 3),
        "rf_auc": round(r["random_forest"]["roc_auc"], 3),
    })

table_a = pd.DataFrame(rows)
print(table_a.round(3).to_string(index=False))
print()
print("prec@50 as points above the test base rate (random ranking):")
for _, row in table_a.iterrows():
    print(f"  {row['config']:30s} rule {row['rule_p50']-row['base_rate']:+.3f} | tree {row['tree_p50']-row['base_rate']:+.3f} | rf {row['rf_p50']-row['base_rate']:+.3f}")

                       config  n_test  base_rate  rule_p50  rule_auc  tree_p50  tree_auc  lr_p50  rf_p50  rf_auc
 before-1 full-month / random    6520      0.268      0.42     0.540      0.36     0.671    0.30    0.80   0.738
before-2 full-month / grouped    4495      0.342      0.36     0.433      0.48     0.501    0.32    0.22   0.515
 after-1  time-aware / random    6520      0.268      0.42     0.540      0.66     0.794    0.90    0.84   0.825
after-2  time-aware / grouped    4495      0.342      0.36     0.433      0.34     0.661    0.60    0.54   0.695

prec@50 as points above the test base rate (random ranking):
  before-1 full-month / random   rule +0.152 | tree +0.092 | rf +0.532
  before-2 full-month / grouped  rule +0.018 | tree +0.138 | rf -0.122
  after-1  time-aware / random   rule +0.152 | tree +0.392 | rf +0.572
  after-2  time-aware / grouped  rule +0.018 | tree -0.002 | rf +0.198


### Reading the before/after table

Three findings, in honest words:

- **Row-random splits are memorization, not skill.** With full-month features the random-split random forest hit
  precision@50 of 0.80; with time-aware features logistic regression hit 0.90 on the random split. Move to a
  client-grouped split and those collapse (RF 0.80 -> 0.22 on full-month; LR 0.90 -> 0.60 on time-aware). Pages
  of one client sitting in both folds let the model memorize the client — the gap is the memorization a random
  split hides.
- **The Week-5 "tree wins" was partly the window overlap.** The shipped numbers reproduce (tree 0.48 vs rule 0.36
  on the grouped split, +0.14 over its 0.342 base rate). But under time-aware features (no second-half overlap)
  the same grouped split drops the tree to 0.34 — exactly the base rate, i.e. no skill. The tree's March edge
  came substantially from features that contained the label window.
- **The honest single-month winners are LR and RF, not the tree.** On the time-aware + grouped split, logistic
  regression measured 0.60 (base 0.342, +0.26) and random forest 0.54 (+0.20); the rule stayed at 0.36. Every
  number is points above its own base rate — the honest tree is not.

**Time split — the deployment simulation.** Train the same models on the **March** decision moment (time-aware
features), score the **April** decision moment (time-aware features). This is the only split that mimics how the
queue is actually used: score today's pages for the next half-month. The April base rate is markedly different
from March's — a reminder that a single-month evaluation is fragile and every score needs its own base rate. A
second row combines the two honest designs: hold out a group of clients from the March training, then score only
those clients' April pages (never seen by the model at all).

In [7]:
# Full time split: train on March moment, test on April moment.
time_results = {}
Xtr_time = build_honest(march)
Xte_time = build_honest(april)
ytr_time = march["is_declining_label"].astype(int).to_numpy()
yte_time = april["is_declining_label"].astype(int).to_numpy()

time_results["time_full"] = {"n_test": len(april), "base_rate": float(yte_time.mean())}
rule_april = encode_rule(april.copy())["queue_score"].to_numpy()
time_results["time_full"]["rule"] = evaluate(yte_time, rule_april)
for name, model in make_models().items():
    model.fit(Xtr_time, ytr_time)
    time_results["time_full"][name] = evaluate(yte_time, model.predict_proba(Xte_time)[:, 1])

# Time + grouped: hold out clients from March train, score only their April pages.
rng = np.random.default_rng(RANDOM_STATE)
clients = np.sort(march["client_hash_id"].drop_duplicates().to_numpy())
n_hold = max(1, int(round(len(clients) * 0.25)))
holdout = set(clients[rng.permutation(len(clients))[:n_hold]])
mtr_mask = ~march["client_hash_id"].isin(holdout).to_numpy()
ate_mask = april["client_hash_id"].isin(holdout).to_numpy()
aptr = march.loc[mtr_mask].reset_index(drop=True)
apte = april.loc[ate_mask].reset_index(drop=True)
yte_apte = apte["is_declining_label"].astype(int).to_numpy()

time_results["time_grouped"] = {"n_test": len(apte), "base_rate": float(yte_apte.mean())}
time_results["time_grouped"]["rule"] = evaluate(yte_apte, encode_rule(apte.copy())["queue_score"].to_numpy())
Xte_apte = build_honest(apte)
for name, model in make_models().items():
    model.fit(build_honest(aptr), aptr["is_declining_label"].astype(int).to_numpy())
    time_results["time_grouped"][name] = evaluate(yte_apte, model.predict_proba(Xte_apte)[:, 1])


def render_time(key, name):
    r = time_results[key]
    line = "  ".join(f"p@{k}={r[name][f'prec@{k}']:.3f}" for k in KS)
    print(f"{name:22s} {line}   auc={r[name]['roc_auc']:.3f} ap={r[name]['avg_precision']:.3f}")

for key, title in [("time_full", "A) train MARCH -> test APRIL (all clients)"),
                   ("time_grouped", "B) train MARCH (25% clients held out) -> test APRIL on held-out clients")]:
    r = time_results[key]
    print(title)
    print(f"  n_test={r['n_test']:,}  base_rate={r['base_rate']:.3f}")
    render_time(key, "rule")
    render_time(key, "logistic_regression")
    render_time(key, "decision_tree_d3")
    render_time(key, "random_forest")
    print()

A) train MARCH -> test APRIL (all clients)
  n_test=36,262  base_rate=0.479
rule                   p@10=0.200  p@25=0.200  p@50=0.240  p@100=0.250   auc=0.478 ap=0.442
logistic_regression    p@10=0.700  p@25=0.680  p@50=0.640  p@100=0.530   auc=0.648 ap=0.562
decision_tree_d3       p@10=0.300  p@25=0.240  p@50=0.300  p@100=0.300   auc=0.634 ap=0.553
random_forest          p@10=0.300  p@25=0.200  p@50=0.240  p@100=0.260   auc=0.646 ap=0.556

B) train MARCH (25% clients held out) -> test APRIL on held-out clients
  n_test=5,195  base_rate=0.507
rule                   p@10=0.000  p@25=0.160  p@50=0.260  p@100=0.280   auc=0.496 ap=0.488
logistic_regression    p@10=0.700  p@25=0.680  p@50=0.700  p@100=0.680   auc=0.710 ap=0.650
decision_tree_d3       p@10=0.600  p@25=0.640  p@50=0.660  p@100=0.630   auc=0.718 ap=0.647
random_forest          p@10=0.700  p@25=0.680  p@50=0.600  p@100=0.610   auc=0.699 ap=0.634



### Reading the time split (the deployment test)

- **The rule fails on April.** Trained on March and scored on April, the rule's precision@50 is 0.24 against an
  April base rate of 0.48 — below random. The w04/05 "keep the rule as the fallback layer" plan does not survive
  deployment.
- **The tree fails the full-window time split but holds on held-out clients — split-dependent, not a flat "does
  not transfer."** On the full April window (split A) w05's pick scores 0.30 against base 0.48 — below random.
  But on the stricter held-out-client time split (B) it measures precision@50 of 0.66 (base 0.51) with the
  highest ROC-AUC of the row (0.718). The honest read is not "the tree is dead" but: its March edge did not
  generalize to the full forward window, and its B number is one slice — treat any tree advantage as a
  hypothesis to re-test, never a settled win.
- **Logistic regression is the most consistent winner across deployment designs.** LR measured precision@50 of
  0.64 on the full April window (base 0.48, +0.16) and 0.70 on the never-seen held-out clients (+0.19 over their
  0.51 base), with ROC-AUC 0.65-0.71 — the only model above random on *both* time-split designs (the forest was
  above random in B at 0.60 but fell to 0.24 in A). The month-to-month swing in the base rate (0.27 -> 0.48) is why a
  single-month evaluation is fragile and every claim must carry its base rate.
- **The honest conclusion partially reverses w05's.** w05 recommended the tree and kept the rule as fallback.
  Honest validation (time-aware features + time split) says the rule is not decision-ready and the tree's
  advantage did not survive forward; logistic regression is the one model whose ranking held on both time-split
  designs. This is the audit doing its job: the claims changed because the evidence changed, not because the
  numbers were hidden.

### Failure examples — what the honest model gets wrong

The honest question for any queue: when the model is wrong, *what kind of wrong is it?* I take the model that
survived both honest designs — logistic regression with time-aware features — and inspect its real errors on the
client-grouped holdout (the after-2 test rows), then on the deployment window (trained on March, scored on April).
Concrete rows are shown with metrics only (no IDs, no client names).

In [8]:
m_tr, m_te, _ = client_grouped_split(march)
tr, te = split_frame(march, m_tr, m_te)
Xtr, Xte = build_honest(tr), build_honest(te)
ytr = tr["is_declining_label"].astype(int).to_numpy()
yte = te["is_declining_label"].astype(int).to_numpy()

lr = make_models()["logistic_regression"]
lr.fit(Xtr, ytr)
proba = lr.predict_proba(Xte)[:, 1]

te = te.copy()
te["lr_proba"] = proba
top = te.sort_values("lr_proba", ascending=False).head(50)
fp = top[top["is_declining_label"] == 0]

half = len(te) // 2
bottom_half = te.sort_values("lr_proba").head(half)
fn = bottom_half[bottom_half["is_declining_label"] == 1]

print(f"Grouped holdout, LR time-aware: top-50 precision = {top['is_declining_label'].mean():.3f} "
      f"(test base rate {yte.mean():.3f}); {len(fp)} of 50 flagged pages did NOT decline; "
      f"{len(fn)} declining pages were ranked in the bottom half (missed by the queue).")

prof = []
for name, grp in [("top-50 queue (all)", top), ("top-50 false positives", fp),
                  ("missed declines (bottom half)", fn), ("test set (all)", te)]:
    if len(grp) == 0:
        continue
    prof.append({
        "group": name,
        "n": len(grp),
        "decline_rate": round(grp["is_declining_label"].mean(), 3),
        "avg_imp_first_half": round(grp["gsc_impressions_first_half"].mean(), 0),
        "avg_pos_first_half": round(grp["avg_position_first_half"].mean(), 1),
        "avg_age_days": round(grp["content_age_days"].mean(), 0),
        "avg_days_since_update": round(grp["days_since_last_update"].mean(), 0),
    })
print(pd.DataFrame(prof).to_string(index=False))

cols = ["gsc_impressions_first_half", "avg_position_first_half", "content_age_days",
        "days_since_last_update", "lr_proba", "is_declining_label"]
print("\nFalse positives — queue says decline, page held (top-50):")
print(fp[cols].head(3).round(2).to_string(index=False))
print("\nMissed declines — declining pages the model ranked in the bottom half:")
print(fn[cols].head(3).round(2).to_string(index=False))

april2 = april.copy()
lr_march = make_models()["logistic_regression"].fit(
    build_honest(march), march["is_declining_label"].astype(int).to_numpy())
april2["lr_proba"] = lr_march.predict_proba(build_honest(april))[:, 1]
top_apr = april2.sort_values("lr_proba", ascending=False).head(50)
fp_apr = top_apr[top_apr["is_declining_label"] == 0]
print(f"\nDeployment (train March -> score April): top-50 precision = {top_apr['is_declining_label'].mean():.3f} "
      f"(April base rate {april2['is_declining_label'].mean():.3f}); {len(fp_apr)} of 50 flagged but did not decline.")

Grouped holdout, LR time-aware: top-50 precision = 0.600 (test base rate 0.342); 20 of 50 flagged pages did NOT decline; 583 declining pages were ranked in the bottom half (missed by the queue).
                        group    n  decline_rate  avg_imp_first_half  avg_pos_first_half  avg_age_days  avg_days_since_update
           top-50 queue (all)   50         0.600             22059.0                18.6         373.0                   18.0
       top-50 false positives   20         0.000             28173.0                12.7         362.0                    NaN
missed declines (bottom half)  583         1.000               162.0                 6.3         230.0                   17.0
               test set (all) 4495         0.342               983.0                 9.4         273.0                   18.0

False positives — queue says decline, page held (top-50):
 gsc_impressions_first_half  avg_position_first_half  content_age_days  days_since_last_update  lr_proba  is_declini


Deployment (train March -> score April): top-50 precision = 0.640 (April base rate 0.479); 18 of 50 flagged but did not decline.


## 3. Leakage audit

The same hunt from Week 3, on the **final** (time-aware) feature set. The attack checklist, run in order:

- [ ] **Timeline drawn** — every feature strictly before the label window (below)
- [ ] **No label-derived or sibling columns** — deliberate leak injection test (the "watch the score jump toward
      1.0" confession test), then removed
- [ ] **No product flags / existing-system scores** as features — column scan
- [ ] **Split grouped by client and/or time** — done in Section 2 (both)
- [ ] **Base rate printed next to every metric** — done everywhere
- [ ] **Top feature importance sanity-checked** — permutation importance on the honest split
- [ ] **Metrics recomputed out-of-fold** — every number above is out-of-fold by construction

### Timeline

Decision moment = **Mar 15**. Features are aggregates over **Mar 1-15** (knowable at the moment). The label
compares **Mar 16-31** against **Mar 1-15** — so the label outcome lives strictly *after* the feature window.
The Week-5 'as-shipped' features instead aggregate over the whole month and therefore *contain* the second-half
window that defines the label: that is the window-overlap leak the time-aware version removes. One more
forward-looking edge: the as-shipped `days_since_last_update` clamped an update that happened *after* the
decision date to 0 ("updated today"), so the honest version treats a post-decision or missing update date as
unknown (`has_days_since_update=0`).

In [9]:
used = set(build_honest(march).columns)
label_window_cols = {"imp_second_half", "imp_first_half", "gsc_impressions_month",
                     "avg_position_month", "sessions_month", "engaged_sessions_month"}
overlap = used & label_window_cols
print("Time-aware feature columns:", sorted(used))
print("Label-window / second-half columns present in features:", sorted(overlap))
print("Timeline is honest (features strictly before the label window):", len(overlap) == 0)

Time-aware feature columns: ['avg_position_first_half', 'content_age_days', 'days_since_last_update', 'engagement_rate_first_half', 'has_days_since_update', 'has_engagement_first_half', 'has_position_first_half', 'log_gsc_impressions_first_half']
Label-window / second-half columns present in features: []
Timeline is honest (features strictly before the label window): True


### The forward-dated `days_since_last_update` clamp — before/after

The second forward-looking edge the audit found is smaller than the window overlap and lives in a
dimension-table date, not a metric window: the as-shipped feature clamped an update that happened
*after* the decision moment to 0, so a page refreshed Mar 20 looked "updated today" to a Mar-15 scorer.
The honest version treats a post-decision or missing update date as unknown (`has_days_since_update=0`).
Same lever as the window overlap, and its measurable cost shows on the deployment split (March-trained
model scored on April) — the only design where a post-decision update can even exist:

In [10]:
def build_honest_clamped(df):
    """The as-shipped update-date semantics on the time-aware features: a post-decision (or
    missing) update date is clamped to 0 = 'updated today', with no has_days_since_update flag
    to say otherwise. Reproduces the w05 clamp this audit removed, on the same first-half features."""
    X = pd.DataFrame(index=df.index)
    X["content_age_days"] = df["content_age_days"].astype(float)
    X["days_since_last_update"] = df["days_since_last_update"].astype(float).fillna(0.0)
    X["avg_position_first_half"] = df["avg_position_first_half"].astype(float).fillna(0.0)
    X["has_position_first_half"] = df["avg_position_first_half"].notna().astype(int)
    X["log_gsc_impressions_first_half"] = np.log1p(df["gsc_impressions_first_half"].astype(float))
    er = pd.to_numeric(df["engaged_sessions_first_half"] / df["sessions_first_half"].where(df["sessions_first_half"] > 0),
                       errors="coerce") * 100.0
    X["has_engagement_first_half"] = er.notna().astype(int)
    X["engagement_rate_first_half"] = er.fillna(0.0)
    return X[["content_age_days", "days_since_last_update", "avg_position_first_half",
              "has_position_first_half", "log_gsc_impressions_first_half",
              "engagement_rate_first_half", "has_engagement_first_half"]]


def lr_deploy_prec50(builder, train_df=march, test_df=april, k=50):
    """Train LR on the March decision moment, score the April decision moment; precision@k."""
    lr = make_models()["logistic_regression"]
    lr.fit(builder(train_df), train_df["is_declining_label"].astype(int).to_numpy())
    y = test_df["is_declining_label"].astype(int).to_numpy()
    return precision_at_k(y, lr.predict_proba(builder(test_df))[:, 1], k)


lr_clamped_p50 = lr_deploy_prec50(build_honest_clamped)
lr_honest_p50 = lr_deploy_prec50(build_honest)
print(f"LR  April deployment  as-shipped clamp (post-decision update = 'updated today'): prec@50={lr_clamped_p50:.3f}")
print(f"LR  April deployment  honest update-date (post-decision update = unknown):       prec@50={lr_honest_p50:.3f}")
print(f"delta (honest - clamped): {lr_honest_p50 - lr_clamped_p50:+.3f}")
print("Verdict:", "the clamp was a real forward-looking leak - the honest number is the lower one"
      if lr_clamped_p50 > lr_honest_p50 else "no measured difference - keep the honest version for principle")

LR  April deployment  as-shipped clamp (post-decision update = 'updated today'): prec@50=0.760
LR  April deployment  honest update-date (post-decision update = unknown):       prec@50=0.640
delta (honest - clamped): -0.120
Verdict: the clamp was a real forward-looking leak - the honest number is the lower one


### The deliberate leak injection test

The skill's own verification: *deliberately ADD a leaky feature and watch the score jump toward 1.0 — if it
doesn't, your test harness itself is broken.* The label is `second_half < first_half`, so the sibling column
`imp_trend_pct = (second - first) / first` carries the answer up to a sign flip. I add it (plus the two raw
halves) to the honest matrix, refit the tree on the same client-grouped split, and watch precision@50. Then I
remove it and keep the honest number. This proves both that the harness detects leaks and that the shipped
feature set does not contain one.

In [11]:
Xh = build_honest(march)
yh = march["is_declining_label"].astype(int).to_numpy()

m_tr, m_te, _ = client_grouped_split(march)

leak = Xh.copy()
leak["imp_first_half"] = march["imp_first_half"].astype(float)
leak["imp_second_half"] = march["imp_second_half"].astype(float)
leak["imp_trend_pct"] = 100.0 * (march["imp_second_half"].astype(float) - march["imp_first_half"].astype(float)) \
    / march["imp_first_half"].astype(float).replace(0, np.nan)
leak = leak.fillna(0.0)  # pages with a zero first half have no trend; the honest set never contains these columns


def tree_scores(X, tr_mask, te_mask):
    Xtr, Xte = X.loc[tr_mask], X.loc[te_mask]
    ytr, yte = yh[tr_mask], yh[te_mask]
    t = DecisionTreeClassifier(class_weight="balanced", max_depth=3, min_samples_leaf=50, random_state=RANDOM_STATE)
    t.fit(Xtr, ytr)
    proba = t.predict_proba(Xte)[:, 1]
    return {"prec@10": precision_at_k(yte, proba, 10), "prec@50": precision_at_k(yte, proba, 50),
            "roc_auc": roc_auc_score(yte, proba)}


honest_score = tree_scores(Xh, m_tr, m_te)
leaky_score = tree_scores(leak, m_tr, m_te)

print(f"tree  honest features : prec@10={honest_score['prec@10']:.3f}  prec@50={honest_score['prec@50']:.3f}  auc={honest_score['roc_auc']:.3f}")
print(f"tree  + label siblings : prec@10={leaky_score['prec@10']:.3f}  prec@50={leaky_score['prec@50']:.3f}  auc={leaky_score['roc_auc']:.3f}")
print()
print("Confession test:", "PASS - harness detects the leak (score jumps toward 1.0)"
      if leaky_score["prec@50"] > 0.9 else "FAIL - harness did NOT detect the leak; investigate")
print("Shipped feature set clean:", honest_score["prec@50"] < 0.7,
      "(honest number is nowhere near the leaky number)")

tree  honest features : prec@10=0.200  prec@50=0.340  auc=0.661
tree  + label siblings : prec@10=1.000  prec@50=1.000  auc=1.000

Confession test: PASS - harness detects the leak (score jumps toward 1.0)
Shipped feature set clean: True (honest number is nowhere near the leaky number)


### Train-without test on the top honest feature

The label is `second_half < first_half`, so first-half impressions is the label's *denominator* — knowable before
the outcome, and a legal feature (like `impressions_prev_30d` in the starter data). The train-without check asks
the honest question: does the score collapse when the volume feature is removed? A collapse would be a
confession; a modest, explainable change is not.

In [12]:
base = tree_scores(Xh, m_tr, m_te)
no_vol = Xh.drop(columns=["log_gsc_impressions_first_half"])
no_vol_score = tree_scores(no_vol, m_tr, m_te)

print(f"tree with    log_gsc_impressions_first_half : prec@50={base['prec@50']:.3f}  auc={base['roc_auc']:.3f}")
print(f"tree without log_gsc_impressions_first_half : prec@50={no_vol_score['prec@50']:.3f}  auc={no_vol_score['roc_auc']:.3f}")
delta = base["prec@50"] - no_vol_score["prec@50"]
print(f"delta prec@50: {delta:+.3f}")
print("Verdict:", "no collapse - the volume feature is a legal signal, not a leak" if abs(delta) < 0.15
      else "score moved a lot - investigate the feature's role")

tree with    log_gsc_impressions_first_half : prec@50=0.340  auc=0.661
tree without log_gsc_impressions_first_half : prec@50=0.380  auc=0.585
delta prec@50: -0.040
Verdict: no collapse - the volume feature is a legal signal, not a leak


### No product flags / existing-system scores

Decision-derived features (product flags, existing scores) would mean learning the old rule, not the world. The
warehouse slice carries raw GSC/GA4 metrics plus metadata — no internal `healthy`/`zombie`-style flags, no
composite score, no trend columns. The scan confirms it, and it lists the label-sibling columns I keep out of
`build_honest` on purpose.

In [13]:
flag_terms = ["flag", "score", "health", "optimization", "zombie", "tier", "trend"]
cols = sorted(march.columns)
print("Columns in the slice:", cols)
flagged = [c for c in cols if any(t in c.lower() for t in flag_terms)]
print("\nAny product-flag / score / trend / tier columns present:", flagged if flagged else "none")
print("Label-sibling columns deliberately excluded from features:",
      [c for c in cols if c in ("imp_first_half", "imp_second_half")])
print("\nNo decision-derived or label-derived columns in the feature set:", not flagged)

Columns in the slice: ['avg_position_first_half', 'avg_position_month', 'client_hash_id', 'content_age_days', 'content_hash_id', 'days_since_last_update', 'engaged_sessions_first_half', 'engaged_sessions_month', 'gsc_impressions_first_half', 'gsc_impressions_month', 'imp_first_half', 'imp_second_half', 'is_declining_label', 'sessions_first_half', 'sessions_month']

Any product-flag / score / trend / tier columns present: none
Label-sibling columns deliberately excluded from features: ['imp_first_half', 'imp_second_half']

No decision-derived or label-derived columns in the feature set: True


### Top-feature sanity (permutation importance, honest split)

Permutation importance on the honest split: a leaky feature carries the label, so shuffling it collapses the score
toward chance (0.5 ROC-AUC) — a near-total collapse on a single feature is the "too good" alarm. Expected here:
nothing near-perfect — the March features are weak demand/position metadata, and the honest split's score sits
modestly above its base rate.

In [14]:
from sklearn.inspection import permutation_importance

Xtr_h = Xh.loc[m_tr]
Xte_h = Xh.loc[m_te]
ytr_h, yte_h = yh[m_tr], yh[m_te]
rf_h = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
                              n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)
rf_h.fit(Xtr_h, ytr_h)
perm = permutation_importance(rf_h, Xte_h, yte_h, n_repeats=10, scoring="roc_auc", random_state=RANDOM_STATE)
perm_df = pd.DataFrame({"feature": Xte_h.columns, "test_auc_drop_when_shuffled": perm.importances_mean,
                        "std": perm.importances_std}).sort_values("test_auc_drop_when_shuffled", ascending=False)
print(perm_df.round(4).to_string(index=False))
worst = perm_df["test_auc_drop_when_shuffled"].max()
print(f"\nLargest AUC drop from shuffling one feature: {worst:.3f}")
print("Verdict:", "no leak - no single feature carries the label" if worst < 0.2
      else "suspicious - one feature carries nearly all the signal; investigate")

                       feature  test_auc_drop_when_shuffled    std
log_gsc_impressions_first_half                       0.1290 0.0075
        days_since_last_update                       0.0097 0.0019
       avg_position_first_half                       0.0092 0.0037
         has_days_since_update                       0.0046 0.0016
    engagement_rate_first_half                       0.0027 0.0014
       has_position_first_half                       0.0020 0.0028
     has_engagement_first_half                       0.0005 0.0011
              content_age_days                      -0.0060 0.0034

Largest AUC drop from shuffling one feature: 0.129
Verdict: no leak - no single feature carries the label


### Leakage audit — summary

Run down the checklist against what the cells above showed:

- Timeline: honest — features are Mar 1-15, label outcome is Mar 16-31 (the shipped full-month features' overlap
  is the one window leak, and it is removed in the time-aware version).
- Label-siblings: the deliberate `imp_trend_pct` injection pushed the tree to ~1.0 precision — the harness
  works — and the shipped set sits nowhere near that, so no sibling column leaked.
- Product flags / existing scores: none in the slice; confirmed by scan.
- Split: client-grouped **and** time-based (Section 2); base rate next to every metric.
- Forward-dated dim dates: the as-shipped `days_since_last_update` clamped post-decision updates to 0 ("updated
  today"); the time-aware set treats them as unknown (`has_days_since_update=0`), so no feature reads a date after
  the decision moment — the measured before/after above cut the deployment-split LR precision@50 from 0.76 to 0.64.
- Top-feature importance: permutation check on the honest split shows nothing whose removal collapses the score
  toward chance — the largest single-feature AUC drop was 0.129 (first-half impressions), expected for the
  dominant signal, not a label carrier.
- All metrics are out-of-fold by construction.

**Verdict: no leakage found in the final (time-aware) feature set.** The genuine leaks in the pipeline were in the
Week-5 shipped features — the full-month window overlap and the post-decision update clamp — neither a label
column, both forward-looking, and both corrected in the time-aware set.

## 4. Claim rewrite

The boldest sentences from my own Week-5 notebook, rewritten in language the evidence can carry. Each rewrite
names the evidence, the design, and what it does *not* say. Numbers are the measured ones from the runs above.

### 4.1 "The depth-3 decision tree is the only model that wins at the top of the queue"


| Before (w05) | Why it overreaches | After (safe) |
|---|---|---|
| *"The depth-3 decision tree is the only model that wins at the top of the queue, and it wins with almost no complexity."* | "Wins" implied a persistent advantage; the evidence was one seed-42 holdout (+0.12 vs the rule). Under time-aware features the edge vanished (0.34 vs a 0.342 base rate), on the full April time split the tree measured 0.30 against a 0.479 base rate (below random), and on the held-out-client time split it measured 0.66 (base 0.51) — split-dependent, not persistent. | *On the March-2026 slice, the depth-3 tree measured precision@50 of 0.48 on the seed-42 client holdout (rule 0.36), but that edge was tied to a feature window that overlapped the label: with time-aware features the same split gave 0.34 (base rate 0.342), the full April time split gave 0.30 (base rate 0.479), and the held-out-client time split gave 0.66 (base rate 0.507). The tree's advantage is split-dependent and not decision-ready; treat any tree edge as a hypothesis to re-test on each new window, not a settled win.* |
| *"Build the queue with the depth-3 decision tree; keep the rule as the explainable reason-codes layer."* | Prescribed an action from a single-month result; the honest time split shows the rule below the April base rate (0.24 vs 0.48) and the tree's March edge not holding forward (0.30 full window; 0.66 on held-out clients). | *Given the measured results, the rule is not ready to drive the queue and the tree's March edge did not hold on the full deployment window. Logistic regression is the model whose ranking held on both honest time-split designs (0.60 March grouped; 0.64 full-window April, 0.70 held-out clients) — a defensible next step is to trial an LR-ranked queue alongside the current process and compare observed precision over a real refresh cycle (decision-support, not a guarantee).* |

### 4.2 "The forest is a ranking refinement" / "the rule weakens on unseen clients"


| Before (w05) | Why it overreaches | After (safe) |
|---|---|---|
| *"Where the forest pays is ranking: its mean test ROC-AUC is the best of the group."* | ROC-AUC on one month is a within-sample ranking property; on the April time split the forest's ROC-AUC (0.646) was indistinguishable from logistic regression's (0.648), so "best" did not survive forward, and its precision@50 swung 0.24 (full April window) to 0.60 (held-out clients) — split-dependent, not a stable edge. | *Across the five March client-grouped splits the random forest measured the highest mean ROC-AUC (0.578) of the models compared; on the April time split its ROC-AUC (0.646) matched logistic regression (0.648), while its precision@50 was 0.24 (full window) and 0.60 (held-out clients). These are observed ranking properties within these slices; whether the ranking improves queue decisions was not measured.* |
| *"The rule weakens on unseen clients — modestly, but really."* | Mostly careful already; the honest addition is the base-rate context and the month-to-month swing. | *The rule's precision@50 was 0.58 on the train clients and 0.36 on the held-out March clients, against a test base rate of 0.342 — about +2 points over random ranking in March. On the April time split it measured 0.24 against a 0.479 base rate — below random. Its full-slice number (about 0.56, w04) was population-local, not a transferable skill.* |

### 4.3 Why safe language matters — the base-rate discipline

Every headline in this audit sits next to its base rate, because a score without its base rate is noise with a
decimal. The single clearest demonstration from this notebook: the **time split** — the model trained on March
(base rate 0.27) is scored on April, where the base rate is 0.48. A model that merely "agreed with March" would
look wrong on April no matter how well it fit March. Claims about this queue must therefore be **observed** (this
slice, this period), **measured** (a comparison on the same split), and **directional / decision-support** (helps
prioritize review), never causal.

## Self-check

Confirmed honestly:

- [x] **Two paper findings named** (Finding #4 Freshness Multiplier; ML-appendix growth classifier) with the
      methodology question for each — label origin and validation design — framed as how to make it stronger
- [x] **My model re-run under an honest split** — time-aware features (before/after) plus a genuine time split
      (train March -> test April), and the row-random numbers so the memorization gap is visible
- [x] **Leakage audit on the final feature set** — timeline drawn, deliberate leak injection (score jumps to ~1.0,
      harness confirmed), train-without test, product-flag scan, permutation-importance sanity, base rate everywhere
- [x] **Real failure examples inspected** — false positives and missed declines of the surviving (LR) model on the
      honest grouped holdout and the April deployment window, with concrete metric-only rows
- [x] **Claims rewritten** in observed / measured / directional / decision-support language, with the evidence that
      carries each one named
- [x] **No client names, URLs, or private queries** anywhere; no datasets committed
- [x] **Runs top to bottom**; metrics receipt at `work/outputs/w06_validation_audit_metrics.json`


### Reading the numbers — what the audit changed about what I believe

1. **The w05 headline reversed.** Week 5 concluded "build the queue with the depth-3 tree, keep the
   rule as fallback." Honest validation says the opposite: under time-aware features + a time split,
   the tree's March edge did not hold forward (full April window 0.30 vs 0.48 base; 0.66 on held-out
   clients — split-dependent), the rule (0.24 vs 0.48 base) is below random, and logistic regression
   (0.60 March grouped / 0.64 April time split / 0.70 time + grouped) is the only consistent winner
   across designs.
2. **Memorization was real and it was hiding in the split.** Row-random splits reached 0.80-0.90
   precision@50; client-grouped splits cut that in half. The reference pipeline's and the paper's
   row-random design would have certified a model that cannot serve unseen clients.
3. **The window overlap was a genuine (small) leak.** The shipped full-month features contained the
   label window; removing that overlap erased the tree's edge. The deliberate leak-injection test
   (adding the label's siblings pushed the tree to precision@50 of 1.000, AUC 1.000) proves the
   harness detects real leaks, and the final time-aware feature set is clean. Removing the forward-dated
   update clamp (Section 3, before/after cell) cut the April LR precision@50 from 0.76 to 0.64 — the second leak
   was real and cost ~12 points on the deployment split.
4. **Base rate is the whole discipline.** The same model, two months, base rates 0.27 and 0.48 — no
   claim about this queue survives without its base rate. Safe language everywhere: observed,
   measured, directional, decision-support.
5. **The honest model's errors are concentrated in the two extremes — and both are low-cost.** On the
   grouped holdout, 20 of the LR queue's top-50 flags were false positives (high-volume pages, avg 28K
   first-half impressions, whose second-half dip did not cross the label threshold; 18 of 50 on April). On the
   grouped holdout the flagged-but-held pages were mostly ones whose update date fell *after* the decision moment
   (unknown at scoring time, so their later refresh was invisible to the queue).
   The 583 declining pages ranked in the bottom half are mostly low-volume (avg 162 first-half
   impressions vs 983 test-wide) whose absolute drops are small. For a refresh queue, the failure worth
   watching is flag precision on high-volume pages; the missed low-volume declines cost little refresh
   value.


In [15]:
receipt = {
    "notebook": "w06_validation_audit",
    "paper_findings": {
        "A_freshness_multiplier": "label = impressions trend + health composite; refreshed set was chosen, not random",
        "B_growth_classifier": "label shares family with features (descriptive); 71% accuracy vs ~62% majority class in the broader portfolio; split grouping not disclosed",
    },
    "slice": "March + April 2026 page-level (w03 contract population)",
    "march_rows": int(len(march)),
    "april_rows": int(len(april)),
    "before_after": table_a.round(3).to_dict("records"),
    "time_split": {
        key: {
            "n_test": r["n_test"],
            "base_rate": round(r["base_rate"], 3),
            "rule_prec50": round(r["rule"]["prec@50"], 3),
            "tree_prec50": round(r["decision_tree_d3"]["prec@50"], 3),
            "tree_auc": round(r["decision_tree_d3"]["roc_auc"], 3),
            "rf_prec50": round(r["random_forest"]["prec@50"], 3),
            "rf_auc": round(r["random_forest"]["roc_auc"], 3),
        } for key, r in time_results.items()
    },
    "leak_injection": {
        "honest_tree_prec50": round(honest_score["prec@50"], 3),
        "leaky_tree_prec50": round(leaky_score["prec@50"], 3),
        "leaky_auc": round(leaky_score["roc_auc"], 3),
    },
    "forward_dated_clamp": {
        "as_shipped_clamp_apr_lr_prec50": round(lr_clamped_p50, 3),
        "honest_update_date_apr_lr_prec50": round(lr_honest_p50, 3),
    },
    "leakage_verdict": "no leakage in final (time-aware) feature set; full-month window overlap in w05 shipped features was the one window leak",
    "failure_examples": {
        "grouped_holdout_top50_prec": round(float(top["is_declining_label"].mean()), 3),
        "grouped_holdout_fp_share_top50": round(float(len(fp) / 50), 3),
        "grouped_holdout_missed_declines_bottom_half": int(len(fn)),
        "deployment_top50_prec": round(float(top_apr["is_declining_label"].mean()), 3),
        "deployment_fp_share_top50": round(float(len(fp_apr) / 50), 3),
    },
    "output": "work/outputs/w06_validation_audit_metrics.json",
}

with open(OUT_DIR / "w06_validation_audit_metrics.json", "w") as f:
    json.dump(receipt, f, indent=2, sort_keys=True)
print("Metrics receipt written to work/outputs/w06_validation_audit_metrics.json")

Metrics receipt written to work/outputs/w06_validation_audit_metrics.json
